# Pandas vs Polars 性能对标测试

基于庞大且带有"杂质"的 `large_data.csv`，对比现代计算引擎的性能差异。

In [3]:
import time
import pandas as pd
import polars as pl

print("库版本信息：")
print(f"Pandas: {pd.__version__}")
print(f"Polars: {pl.__version__}")

库版本信息：
Pandas: 3.0.1
Polars: 1.39.0


## 1. CSV 读取性能测试

In [4]:
# 1. 传统基准：Pandas
start = time.time()
df_pd = pd.read_csv("large_data.csv")
pandas_read_time = time.time() - start
print(f"Pandas 读取耗时：{pandas_read_time:.3f} 秒")
print(f"数据形状：{df_pd.shape}")

Pandas 读取耗时：4.940 秒
数据形状：(1000000, 6)


In [5]:
# 2. 现代多线程：Polars
start = time.time()
df_pl = pl.read_csv("large_data.csv")
polars_read_time = time.time() - start
print(f"Polars 读取耗时：{polars_read_time:.3f} 秒")
print(f"数据形状：{df_pl.shape}")

Polars 读取耗时：0.213 秒
数据形状：(1000000, 6)


In [6]:
# 性能对比
speedup = pandas_read_time / polars_read_time
print(f"\n{'='*50}")
print(f"性能对比结果：")
print(f"{'='*50}")
print(f"Pandas 耗时：{pandas_read_time:.3f} 秒")
print(f"Polars 耗时：{polars_read_time:.3f} 秒")
print(f"Polars 加速比：{speedup:.2f}x")
print(f"时间节省：{((pandas_read_time - polars_read_time) / pandas_read_time * 100):.1f}%")


性能对比结果：
Pandas 耗时：4.940 秒
Polars 耗时：0.213 秒
Polars 加速比：23.22x
时间节省：95.7%


## 2. 数据筛选性能测试

In [7]:
# Pandas 筛选
start = time.time()
result_pd = df_pd[df_pd['event_type'] == 'login']
pandas_filter_time = time.time() - start
print(f"Pandas 筛选耗时：{pandas_filter_time:.3f} 秒")
print(f"筛选结果：{len(result_pd)} 行")

Pandas 筛选耗时：0.146 秒
筛选结果：164446 行


In [8]:
# Polars 筛选
start = time.time()
result_pl = df_pl.filter(pl.col('event_type') == 'login')
polars_filter_time = time.time() - start
print(f"Polars 筛选耗时：{polars_filter_time:.3f} 秒")
print(f"筛选结果：{len(result_pl)} 行")

Polars 筛选耗时：0.072 秒
筛选结果：164446 行


In [9]:
speedup = pandas_filter_time / polars_filter_time
print(f"\n筛选性能对比：Polars 加速 {speedup:.2f}x")


筛选性能对比：Polars 加速 2.03x


## 3. 分组聚合性能测试

In [10]:
# Pandas 分组聚合
start = time.time()
group_pd = df_pd.groupby('event_type').size().reset_index(name='count')
pandas_group_time = time.time() - start
print(f"Pandas 分组聚合耗时：{pandas_group_time:.3f} 秒")
print(group_pd)

Pandas 分组聚合耗时：0.073 秒
   event_type   count
0       LogIn     623
1       ckick     593
2        clic     640
3       click  164046
4        clik     673
5      cllick     629
6     log-out     821
7       login  164446
8      login      639
9        logn     618
10      logot     821
11     logout  164370
12      logut     843
13      loign     617
14    puchase     826
15   purchase  163849
16    purchse     880
17   purhcase     840
18      seach     848
19     search  164499
20     serach     784
21      serch     861
22        vew     848
23      vieew     835
24       view  163749
25       viwe     802


In [11]:
# Polars 分组聚合
start = time.time()
group_pl = df_pl.group_by('event_type').agg(pl.len().alias('count'))
polars_group_time = time.time() - start
print(f"Polars 分组聚合耗时：{polars_group_time:.3f} 秒")
print(group_pl)

Polars 分组聚合耗时：0.058 秒
shape: (26, 2)
┌────────────┬────────┐
│ event_type ┆ count  │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ cllick     ┆ 629    │
│ view       ┆ 163749 │
│ login      ┆ 639    │
│ loign      ┆ 617    │
│ ckick      ┆ 593    │
│ …          ┆ …      │
│ clik       ┆ 673    │
│ purchse    ┆ 880    │
│ serach     ┆ 784    │
│ purchase   ┆ 163849 │
│ log-out    ┆ 821    │
└────────────┴────────┘


In [12]:
speedup = pandas_group_time / polars_group_time
print(f"\n分组聚合性能对比：Polars 加速 {speedup:.2f}x")


分组聚合性能对比：Polars 加速 1.25x


## 4. 复杂查询性能测试（多条件筛选 + 聚合）

In [13]:
# Pandas 复杂查询
start = time.time()
complex_pd = df_pd[
    (df_pd['event_type'] == 'click') | (df_pd['event_type'] == 'purchase')
].groupby('user_id').size().reset_index(name='count')
pandas_complex_time = time.time() - start
print(f"Pandas 复杂查询耗时：{pandas_complex_time:.3f} 秒")
print(f"结果行数：{len(complex_pd)}")

Pandas 复杂查询耗时：0.338 秒
结果行数：228818


In [14]:
# Polars 复杂查询
start = time.time()
complex_pl = df_pl.filter(
    (pl.col('event_type') == 'click') | (pl.col('event_type') == 'purchase')
).group_by('user_id').agg(pl.len().alias('count'))
polars_complex_time = time.time() - start
print(f"Polars 复杂查询耗时：{polars_complex_time:.3f} 秒")
print(f"结果行数：{len(complex_pl)}")

Polars 复杂查询耗时：0.041 秒
结果行数：228818


In [15]:
speedup = pandas_complex_time / polars_complex_time
print(f"\n复杂查询性能对比：Polars 加速 {speedup:.2f}x")


复杂查询性能对比：Polars 加速 8.25x


## 5. 性能汇总

In [17]:
import pandas as pd

summary = pd.DataFrame({
    '操作': ['CSV 读取', '数据筛选', '分组聚合', '复杂查询'],
    'Pandas 耗时 (秒)': [pandas_read_time, pandas_filter_time, pandas_group_time, pandas_complex_time],
    'Polars 耗时 (秒)': [polars_read_time, polars_filter_time, polars_group_time, polars_complex_time],
    '加速比': [
        pandas_read_time / polars_read_time,
        pandas_filter_time / polars_filter_time,
        pandas_group_time / polars_group_time,
        pandas_complex_time / polars_complex_time
    ]
})

print("\n" + "="*60)
print("性能测试汇总")
print("="*60)
print(summary.to_string(index=False))
print("\n" + "="*60)
print(f"平均加速比：{summary['加速比'].mean():.2f}x")
print("="*60)


性能测试汇总
    操作  Pandas 耗时 (秒)  Polars 耗时 (秒)       加速比
CSV 读取       4.939643       0.212776 23.215233
  数据筛选       0.145689       0.071901  2.026249
  分组聚合       0.072931       0.058277  1.251457
  复杂查询       0.338136       0.040998  8.247726

平均加速比：8.69x


## 思考：底层引擎差异带来的"降维打击"

| 特性 | Pandas | Polars |
|------|--------|--------|
| **执行引擎** | 单线程 | 多线程并行 |
| **内存管理** | 复制语义 | 零拷贝优化 |
| **查询优化** | 即时执行 | 查询计划优化 |
| **数据格式** | NumPy 数组 | Arrow 列式存储 |
| **惰性求值** | ❌ | ✅ (lazy API) |

**核心优势：**
1. **SIMD 向量化**：利用 CPU 单指令多数据流特性
2. **多核并行**：自动利用所有可用 CPU 核心
3. **查询融合**：合并多个操作减少中间结果
4. **流式处理**：大数据集无需全部加载到内存